# Create Gulbenkian Foundation Awards (GRANT PATTERN, method-2 WordPress REST API)

Fundação Calouste Gulbenkian projects from the foundation's own WordPress REST API at gulbenkian.pt. Portugal's largest private cultural and scientific philanthropic organization (founded 1956 from Calouste Gulbenkian's estate), funding projects in arts, sciences, education, and social cohesion.

**Prerequisites:**
- Run `scripts/local/gulbenkian_to_s3.py` first.

**Data source:** `https://gulbenkian.pt/wp-json/wp/v2/project` (paginated, ~401 records). ACF fields expose date_start, date_end, local, partners, duration, beneficiaries, budget (typically empty).

**S3 location:** `s3a://openalex-ingest/awards/gulbenkian/gulbenkian_projects.parquet`

**Awarding body:**
- funder_id: 4320323335
- display_name: "Fundação Calouste Gulbenkian"
- country: PT
- ROR: (not exposed in the OpenAlex record)
- DOI: 10.13039/501100005635

The GB-domiciled UK Branch is a separate OpenAlex funder (F4320320052) and is intentionally NOT covered by this ingest.

**Coverage (full local run, 2026-05-22):**
- 401 projects total via the `/wp/v2/project` endpoint
- 100% title / slug / start_year / description
- ~32% partners (text field)
- 0% amount/budget (foundation publishes program-level budgets in narrative form but doesn't populate the ACF `budget` field per-project) — §6.7 waiver
- 2 slug collisions resolved via project_id disambiguation in `funder_award_id`
- 401/401 unique project_id (WP post ID, canonical)

**Amount waiver (§6.7):** When the foundation publishes a budget value in the ACF `budget` field, the script parses it into amount/currency (EUR). On the current corpus 0/401 projects have populated budget, so all rows are NULL. This is a source-side gap, not a contractor-side decision. Precedent for NULL amount with structural budget support: HHMI #44, CIFAR #79, NOMIS #109.

**Provenance:** `gulbenkian_projects` (direct from foundation, not an aggregator).

**Priority:** 114 (UCOP 106 / Rita Allen 107 / Schmidt Sciences 108 / NOMIS 109 / Wenner-Gren 110 / KAW 111 / Helmsley 112 / Mott 113 are immediately-prior slots in flight).


## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gulbenkian_raw
USING delta AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/gulbenkian/gulbenkian_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) as total_rows FROM openalex.awards.gulbenkian_raw;


In [ ]:
%sql DESCRIBE openalex.awards.gulbenkian_raw; 

## Step 1.5: Inspect Raw Data, Money Scan, Native Key

Gulbenkian's ACF budget field is currently empty across the corpus (§6.7 waiver applies). The script captures budget if populated; runbook §1.5 money/currency scan is run for completeness.

In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.gulbenkian_raw)
WHERE LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|fund_|funding|cost|budget|grant_offer|awarded|valeur|monto|importe|montant|betrag|valor|importo|kwota|belopp';


In [ ]:
%sql
SELECT column_name FROM (DESCRIBE openalex.awards.gulbenkian_raw)
WHERE LOWER(column_name) RLIKE 'currenc|ccy|iso_4217';


In [ ]:
%sql
SELECT project_id, slug, title, start_year, partners,
       SUBSTR(description, 1, 200) AS desc_preview
FROM openalex.awards.gulbenkian_raw LIMIT 5;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_ids,
    COUNT(DISTINCT project_id) AS distinct_project_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.gulbenkian_raw;


In [ ]:
%sql
SELECT start_year, COUNT(*) as projects
FROM openalex.awards.gulbenkian_raw
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;


In [ ]:
%sql
-- Coverage of optional ACF fields
SELECT
    COUNT(amount) AS has_amount,
    COUNT(partners) AS has_partners,
    COUNT(local) AS has_local,
    COUNT(duration) AS has_duration,
    COUNT(beneficiaries) AS has_beneficiaries,
    COUNT(date_start) AS has_date_start,
    COUNT(date_end) AS has_date_end,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount
FROM openalex.awards.gulbenkian_raw;


## Step 1.6: Fail-fast — Verify the Gulbenkian Funder Row Exists

**Runbook §2.2.4 mandatory.**

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.common.funder
WHERE funder_id = 4320323335;


## Step 2: Transform to OpenAlex Awards Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.gulbenkian_awards
USING delta
AS
WITH
gulbenkian_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320323335
),

raw_prepared AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS parsed_amount,
        TRY_CAST(start_year AS INT) AS parsed_start_year,
        TRY_CAST(end_year AS INT) AS parsed_end_year,
        TRY_TO_DATE(CONCAT(start_year, '-01-01'), 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(CONCAT(end_year, '-12-31'), 'yyyy-MM-dd') AS parsed_end_date
    FROM openalex.awards.gulbenkian_raw
    WHERE title IS NOT NULL
      AND TRIM(title) != ''
      AND funder_award_id IS NOT NULL
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,

        g.title as display_name,
        g.description as description,

        f.funder_id,
        g.funder_award_id,

        g.parsed_amount as amount,
        g.currency as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,
        'Gulbenkian Project' as funder_scheme,

        'gulbenkian_projects' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        g.parsed_start_year as start_year,
        g.parsed_end_year as end_year,

        -- lead_investigator: Gulbenkian projects are typically run by partner orgs (the partner_name field),
        -- not by an individual PI. Per HHMI/Hewlett/Helmsley/Mott precedent, populate
        -- affiliation.name with the partners string (may include multiple orgs).
        CASE
            WHEN g.partners IS NULL OR TRIM(g.partners) = '' THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                CAST(NULL AS STRING) as given_name,
                CAST(NULL AS STRING) as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    g.partners as name,
                    'PT' as country,  -- most Gulbenkian projects are Portugal-based
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        g.link as landing_page_url,

        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN gulbenkian_funder f
)

SELECT * FROM awards_transformed;


## Step 3: Insert Into `openalex_awards_raw` With Priority 114

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'gulbenkian_projects' AND priority = 114;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    114 as priority  -- Gulbenkian priority (matches CreateAwards.ipynb registry)
FROM openalex.awards.gulbenkian_awards;


## Verification Queries

In [ ]:
%sql SELECT COUNT(*) as total FROM openalex.awards.gulbenkian_awards; 

In [ ]:
%sql DESCRIBE openalex.awards.gulbenkian_awards; 

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator) as has_pi,
    COUNT(landing_page_url) as has_url,
    ROUND(COUNT(display_name) * 100.0 / COUNT(*), 1) as pct_title,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) as pct_amount,
    ROUND(COUNT(description) * 100.0 / COUNT(*), 1) as pct_description,
    ROUND(COUNT(start_date) * 100.0 / COUNT(*), 1) as pct_start_date,
    ROUND(COUNT(lead_investigator) * 100.0 / COUNT(*), 1) as pct_pi
FROM openalex.awards.gulbenkian_awards;


In [ ]:
%sql
SELECT funder_award_id, display_name, start_year, end_year,
       lead_investigator.affiliation.name AS partner_org, landing_page_url
FROM openalex.awards.gulbenkian_awards LIMIT 10;


In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.gulbenkian_awards GROUP BY funder.display_name;


In [ ]:
%sql
SELECT start_year, COUNT(*) as projects
FROM openalex.awards.gulbenkian_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 30;


In [ ]:
%sql
-- §6.7 expected: 0% pct_amount (waived) until source publishes budget per project
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies
FROM openalex.awards.gulbenkian_awards;


In [ ]:
%sql
SELECT
    COUNT(*) AS total,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS collisions
FROM openalex.awards.gulbenkian_awards;
